# Brain Age Batch Inference - Colab GPU

Runs **SFCN**, **SynthBA**, and **MIDIBrainAge** on IXI scans.

**Before running:** Runtime -> Change runtime type -> **GPU (T4)**

| Model | Paper | Preprocessing |
|-------|-------|---------------|
| SFCN | Peng et al. 2021 | N4 + deepbet + MNI affine + crop [160,192,160] |
| SynthBA | Puglisi et al. 2024 | built-in (SynthSeg + ANTs MNI) |
| MIDI | Wood et al. 2024 | built-in (HD-BET + ANTs MNI) |

## 0. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

REPO = "/content/faceage-to-brainage"

if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/YOUR_USERNAME/faceage-to-brainage.git {REPO}")
else:
    os.system(f"git -C {REPO} pull --rebase")

os.chdir(REPO)
print("Working dir:", os.getcwd())

## 1. Install dependencies

In [ ]:
import os
os.system("pip install -q numpy scipy pandas matplotlib tqdm Pillow nibabel scikit-image torch torchvision SimpleITK nilearn deepbet synthba antspyx monai pingouin seaborn")
if not os.path.exists("vendor/SFCN"):
    os.system("git clone --depth 1 https://github.com/ha-ha-ha-han/UKBiobank_deep_pretrain vendor/SFCN")
if not os.path.exists("vendor/MIDIBrainAge"):
    os.system("git clone --depth 1 https://github.com/MIDIconsortium/BrainAge vendor/MIDIBrainAge")
print("done")

## 2. SFCN weights

Copy weights from Drive or provide another path.

In [ ]:
import os, shutil

SFCN_WEIGHT_DST = "vendor/SFCN/brain_age/run_20190719_00_epoch_best_mae.p"
SFCN_WEIGHT_SRC = "/content/drive/MyDrive/brainage_weights/run_20190719_00_epoch_best_mae.p"

if not os.path.exists(SFCN_WEIGHT_DST):
    if os.path.exists(SFCN_WEIGHT_SRC):
        os.makedirs(os.path.dirname(SFCN_WEIGHT_DST), exist_ok=True)
        shutil.copy(SFCN_WEIGHT_SRC, SFCN_WEIGHT_DST)
        print("Copied weights from Drive")
    else:
        print("WARNING: put SFCN weights at:", SFCN_WEIGHT_SRC)
else:
    print("SFCN weights found:", SFCN_WEIGHT_DST)

## 3. Configure paths

**Edit the variables below** to match your Drive layout.

In [ ]:
import json, os
from pathlib import Path

# --- EDIT THESE ---
DRIVE_ROOT   = "/content/drive/MyDrive"
IXI_MANIFEST = f"{DRIVE_ROOT}/IXI/ixi_t1_manifest.csv"
RESULTS_DIR  = f"{DRIVE_ROOT}/brainage_results"
PREPROC_DIR  = f"{DRIVE_ROOT}/brainage_preproc"
REPO         = "/content/faceage-to-brainage"
SFCN_WEIGHT  = f"{REPO}/vendor/SFCN/brain_age/run_20190719_00_epoch_best_mae.p"
# ------------------

os.makedirs(f"{RESULTS_DIR}/ixi", exist_ok=True)
os.makedirs(f"{PREPROC_DIR}/ixi", exist_ok=True)

config = {
    "runtime": {"device": "cuda"},
    "sfcn": {
        "model_dir":          f"{REPO}/vendor/SFCN",
        "weight_path":        SFCN_WEIGHT,
        "skullstrip_command": "deepbet",
        "skip_skullstrip":    False,
        "keep_skullstripped": False,
        "n4_correct":         True,
        "register_mni":       True,
        "age_bins":           {"start": 42.0, "step": 1.0, "count": 40}
    },
    "datasets": {
        "ixi": {
            "manifest_csv":      IXI_MANIFEST,
            "input_path_column": "t1_filename",
            "chron_age_column":  "AGE",
            "subject_id_column": "IXI_ID",
            "output_csv":        f"{RESULTS_DIR}/ixi/predictions.csv",
            "preproc_dir":       f"{PREPROC_DIR}/ixi"
        }
    }
}

cfg_path = f"{REPO}/config/local/brain_age_local.json"
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
Path(cfg_path).write_text(json.dumps(config, indent=2))
print("Config written:", cfg_path)

## 4. Smoke test - 3 scans, all models

In [ ]:
import subprocess, sys

def run_batch(model, tta=False, limit=3, dataset="ixi"):
    cmd = [
        sys.executable, "scripts/batch_brainage.py",
        "--model", model,
        "--dataset", dataset,
        "--config", "config/local/brain_age_local.json",
    ]
    if limit is not None:
        cmd += ["--limit", str(limit)]
    if tta:
        cmd.append("--tta")
    print("
>>> Running:", " ".join(cmd))
    r = subprocess.run(cmd, text=True)
    return r.returncode

run_batch("sfcn", limit=3)

In [ ]:
run_batch("synthba", limit=3)

In [ ]:
run_batch("midi", limit=3)

## 5. Check smoke test results

In [ ]:
import pandas as pd
from pathlib import Path

for csv in sorted(Path(RESULTS_DIR, "ixi").glob("*.csv")):
    df = pd.read_csv(csv)
    ok = (df["status"] == "ok").sum()
    print(f"
{csv.name}  ({ok}/{len(df)} ok)")
    print(df[["subject_id","chron_age","predicted_age","brain_age_gap","status"]].to_string())

## 6. Full batch - all IXI scans

Each model writes its own CSV. GPU runtime: ~30 min total for 561 scans.

In [ ]:
run_batch("sfcn", limit=None)

In [ ]:
run_batch("synthba", limit=None)

In [ ]:
run_batch("midi", limit=None)

## 7. Compare models

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

frames = [pd.read_csv(p) for p in Path(RESULTS_DIR, "ixi").glob("*.csv")]
all_df = pd.concat(frames, ignore_index=True)
ok = all_df[all_df["status"] == "ok"].copy()

stats = ok.groupby("model_name").apply(lambda g: pd.Series({
    "n":    len(g),
    "MAE":  g["brain_age_gap"].abs().mean(),
    "bias": g["brain_age_gap"].mean(),
    "r":    g[["chron_age","predicted_age"]].corr().iloc[0,1]
})).round(2)
print(stats)

models = ok["model_name"].unique()
fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 5))
if len(models) == 1:
    axes = [axes]
for ax, (model, gdf) in zip(axes, ok.groupby("model_name")):
    ax.scatter(gdf["chron_age"], gdf["predicted_age"], alpha=0.4, s=10)
    lims = [gdf["chron_age"].min()-2, gdf["chron_age"].max()+2]
    ax.plot(lims, lims, "k--", lw=0.8)
    ax.set_title(f"{model}
MAE={gdf['brain_age_gap'].abs().mean():.1f} y")
    ax.set_xlabel("Chronological age"); ax.set_ylabel("Predicted age")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/scatter_all_models.png", dpi=150)
plt.show()